<a href="https://colab.research.google.com/github/CooL-Legend/CSCI544-DLM/blob/sean-1105/notebooks/gsm8k/Experiment_1_Eval_IG_KL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>


# Experiment 1 & 3: MEDAL vs. AB-MEDAL on GSM8K — IG + KL Evaluation (20 Questions)

This notebook runs **Experiment 1 (E1)** (Accuracy and Inference Efficiency) and **Experiment 3 (E3)** (Compute Allocation and Tree Topology Analysis) comparing **MEDAL** and **AB-MEDAL** on GSM8K with `LLaDA-8B-Instruct`.

---

## 1. Experimental Design & Controlled Isolation
The experiment isolates exactly one architectural difference:
- **MEDAL**: UCB selection policy with fixed sibling-oriented expansion (up to $K_2=5$ siblings per parent).
- **AB-MEDAL**: Adaptive Bayesian Student-t GEN/CONT selection with individual adaptive expansions.

| Component | MEDAL | AB-MEDAL |
|---|---|---|
| **Backbone Model** | LLaDA-8B-Instruct (bfloat16) | Same |
| **Questions** | Frozen GSM8K (20 questions from `evaluation`) | Same |
| **Prompt** | Two-shot decomposition (pens & muffins) | Same |
| **Search Budgets** | $B \in \{256, 512\}$ NFEs (Conditions A & B) | Same |
| **Max Search Depth** | $L_c = 10$ | Same |
| **Eligible Actions** | $K_1 = 3, K_2 = 5$ | Same |
| **Immediate Reward** | $R = IG(s,s') - \lambda D_{KL}(p_g^i \Vert p_s^i)$; $\lambda=0.05$ | Same |
| **Candidate Selection** | Minimum-$C=3$ rule: $d^* = \max\{d \in [1, L_c]: N_d \ge 3\}$ | Same |
| **Completion Decoder** | Deterministic Confidence at $T=0$ (256 length) | Same |
| **Completion Caching** | Shared partial states decode once | Same |
| **Search Tree Checkpointing** | Saved at exact budgets 256, 512 (Conditions A & B) | Same |

---

### Reward B: branch-local, pre-commit stability

For `g → s → s'`, action `(i,v)` commits token `v` at masked position `i` in `s`.
IG measures the entropy reduction from `s` to `s'`; KL compares the **previous and current predictions at i**, from `g` to `s`, before commitment. Root penalties are zero. Each branch uses its own history, never the last visited sibling.

Both UCB backups and AB-MEDAL Student-t updates receive the combined reward. Search actions, budgets, depth, candidate selection, prompt and completion are unchanged. Default `KL_TOP_K=32` uses the previous top-32 token IDs plus one tail bucket; `0` uses full-vocabulary KL at higher CPU RAM cost. No extra model forwards are needed, but KL computation and transfers add runtime. `KL_WEIGHT=0` reproduces original IG search.

The original notebook remains unchanged. Results use a separate reward-specific directory, and incompatible saved reward settings are rejected instead of resumed. Accuracy and speed improvements are hypotheses, not established results.

## 2. Continuous Search Checkpointing Optimization
Rather than executing two separate searches per question and method ($256 + 512 = 768$ NFEs), search runs continuously to $B=512$ and captures prefix snapshots at $B=256, 512$.

Prefix equivalence ($T_{256}^{(512)} \equiv T_{256}^{(256)}$) is checked with synthetic-model tests and can be verified with the optional real-model preflight. Reusing the prefix saves 256 search NFEs per question and method.
---

## 3. Pure Offline E3 Analysis
E3 reads the saved E1 traces and generates comprehensive compute allocation metrics and all 6 publication figures **without performing any additional model forwards**.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Verify GPU Hardware
Verify that a CUDA GPU supporting bfloat16 (recommended: L4 or A100 with $\ge 22$ GiB VRAM) is active.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select a CUDA GPU runtime in Colab (L4 or A100), then reconnect.")
gpu = torch.cuda.get_device_properties(0)
vram_gib = gpu.total_memory / (1024 ** 3)
print(f"GPU: {gpu.name} | Total VRAM: {vram_gib:.1f} GiB")
if vram_gib < 20:
    raise RuntimeError("This notebook uses unquantized BF16 weights. Choose an L4 or A100 runtime with >= 22 GB VRAM; 16 GB T4 is too small.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("This GPU does not support bfloat16 inference.")
print("✓ GPU hardware verification passed.")

## 2. Repository Setup
Clone or update the GitHub repository on branch `sean-1105` (commit and push this branch before using Colab). Supports private repository authentication via Colab Secrets or password prompt.

In [ ]:
from pathlib import Path
import getpass
import os
import subprocess
import sys
import tempfile

REPOSITORY = "CooL-Legend/CSCI544-DLM"
BRANCH = "sean-1105"
REPO_URL = f"https://github.com/{REPOSITORY}.git"
IN_COLAB = Path("/content").is_dir()
REPO_DIR = Path("/content/CSCI544-DLM") if IN_COLAB else next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/dlm_project").is_dir()
)

def run_git(args, token=None):
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    with tempfile.TemporaryDirectory() as tmp:
        if token:
            askpass = Path(tmp) / "askpass.py"
            askpass.write_text(
                "#!/usr/bin/env python3\nimport os, sys\n"
                "print('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['COLAB_GH_TOKEN'])\n"
            )
            askpass.chmod(0o700)
            env.update(GIT_ASKPASS=str(askpass), COLAB_GH_TOKEN=token)
        return subprocess.run(["git", "-c", "credential.helper=", *args], env=env, capture_output=True, text=True)

def remote_git(args):
    result = run_git(args)
    if result.returncode:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
        if not token:
            token = getpass.getpass("GitHub token with read access to private repo CooL-Legend/CSCI544-DLM: ")
        try:
            result = run_git(args, token)
        finally:
            del token
    if result.returncode:
        raise RuntimeError(result.stderr.strip() or "GitHub clone/update failed.")

def ensure_checkout():
    def local_git(*args):
        return subprocess.check_output(["git", "-C", str(REPO_DIR), *args], text=True).strip()

    if REPO_DIR.exists():
        if not (REPO_DIR / ".git").is_dir():
            raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout.")
        fetch_specs = local_git("config", "--get-all", "remote.origin.fetch").splitlines()
        branch_ref = f"refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}"
        if not any(spec.lstrip("+") in (branch_ref, "refs/heads/*:refs/remotes/origin/*") for spec in fetch_specs):
            local_git("remote", "set-branches", "--add", "origin", BRANCH)
        remote_git(["-C", str(REPO_DIR), "fetch", "origin", branch_ref])
        exists = subprocess.run(["git", "-C", str(REPO_DIR), "show-ref", "--verify", "--quiet", f"refs/heads/{BRANCH}"]).returncode == 0
        if exists:
            local_git("switch", BRANCH)
        else:
            local_git("switch", "--create", BRANCH, "--track", f"origin/{BRANCH}")
        local_git("merge", "--ff-only", f"origin/{BRANCH}")
    else:
        remote_git(["clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
    print(f"Code ready: {REPO_DIR} | branch: {BRANCH} | commit: {local_git('rev-parse', '--short', 'HEAD')}")

if IN_COLAB:
    ensure_checkout()
else:
    print(f"Using local repository checkout: {REPO_DIR}")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

## 3. Install Dependencies
Install pinned LLaDA dependencies, editable package installation, and plotting tools.

In [ ]:
import site
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements/llada-colab.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements/dev.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO_DIR)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib", "pandas", "seaborn"], check=True)

for pkg in [*site.getsitepackages(), site.getusersitepackages()]:
    site.addsitedir(pkg)
print("✓ Dependencies installed and package paths loaded.")

## 4. Run Algorithmic Pre-Flight Unit Tests
Run unit tests with synthetic models to confirm checkpoint prefix equivalence, candidate selection, completion caching, and E3 offline metric accounting before loading weights.

In [ ]:
import subprocess
import sys

res = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/decoding/test_checkpoint_equivalence.py",
     "tests/decoding/test_ig_kl_search.py", "tests/tasks/test_experiment_1.py",
     "tests/tasks/test_experiment_1_ig_kl.py", "tests/tasks/test_e3_analysis.py",
     "tests/workflows/test_e1_ig_kl_notebook.py", "-v"],
    cwd=REPO_DIR, capture_output=True, text=True
)
print(res.stdout)
if res.returncode != 0:
    print("--- PRE-FLIGHT TESTS FAILED ---")
    print(res.stderr)
    raise RuntimeError("Algorithmic tests failed. Review environment before loading weights.")
print("✓ All requested pre-flight algorithmic unit tests passed.")

## 5. Mount Google Drive & Configure Cache
Mount Google Drive for persistent caching of model weights and experimental artifacts across Colab sessions.

In [ ]:
from pathlib import Path
import os

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/CSCI544-DLM")
else:
    DRIVE_ROOT = REPO_DIR / ".cache/e1_e3"

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["HF_HUB_CACHE"] = str(DRIVE_ROOT / "cache/hub")
os.environ["HF_DATASETS_CACHE"] = str(DRIVE_ROOT / "cache/datasets")

OUTPUT_BASE = DRIVE_ROOT / "results/e1_e3_ig_kl"
OUTPUT_DIR = OUTPUT_BASE  # Refined by lambda and top-k in Section 6.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Persistent Drive Root:", DRIVE_ROOT)
print("Experiment Output Directory:", OUTPUT_DIR)

## 6. Experiment Controls & Configuration
Configure the 20-question frozen evaluation set, budgets [256, 512] (Conditions A & B), depth 10, minimum-C=3 rule, and enforce serial execution (batch size 1).

In [ ]:
from dlm_project.tasks.gsm8k.experiment_1 import (
    CHECKPOINT_BUDGETS, MAX_SEARCH_DEPTH, MIN_CANDIDATES, context
)

REWARD_MODE = "ig_kl"
KL_WEIGHT = 0.05  # Initial experimental value; tune on dev, not evaluation.
KL_TOP_K = 32     # Previous top-k + tail; 0 = full-vocabulary KL.

STAGE = "evaluation"
METHODS = ["medal", "ab_medal"]
BUDGETS = list(CHECKPOINT_BUDGETS)  # [256, 512] (Conditions A & B)
MAX_DEPTH = MAX_SEARCH_DEPTH        # 10
MIN_C = MIN_CANDIDATES              # 3
BATCH_SIZE = 1                      # Serial execution strictly enforced

ctx = context(
    stage=STAGE,
    methods=tuple(METHODS),
    budgets=tuple(BUDGETS),
    max_depth=MAX_DEPTH,
    num_candidates=MIN_C,
    reward_mode=REWARD_MODE, kl_weight=KL_WEIGHT, kl_top_k=KL_TOP_K,
)

OUTPUT_DIR = OUTPUT_BASE / f"lambda_{KL_WEIGHT:g}_topk_{KL_TOP_K}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Reward:", ctx["reward"])
print("Reward-specific output:", OUTPUT_DIR)

print(f"=== Experiment Configuration ({STAGE.upper()}) ===")
print("  Protocol:", ctx["protocol_version"])
print(f"  Stage: {ctx['stage']} ({len(ctx['examples'])} frozen questions)")
print("  Methods:", ctx["methods"])
print("  Budgets:", ctx["budgets"])
print("  Max Search Depth (Lc):", ctx["max_depth"])
print("  Candidate Selection Rule: min-C =", ctx["num_candidates"])
print("  Batch Size:", BATCH_SIZE, "(strictly serial)")
print("\nFirst question preview:")
print("  ID:", ctx["examples"][0]["example_id"])
print("  Question:", ctx["examples"][0]["question"])
print("  Answer:", ctx["examples"][0].get("reference_solution", "").split("####")[-1].strip())


## 7. Cache & Load LLaDA-8B-Instruct
Load the pinned `LLaDA-8B-Instruct` model and tokenizer on CUDA in bfloat16, and prepare the tokenized evaluation prompts.

In [ ]:
import shutil
from dlm_project.models.profiles import get_profile
from dlm_project.tasks.gsm8k.experiment_1 import load_model, prepare

profile = get_profile("llada")
model_name = profile["repo_id"].split("/")[-1]
drive_model_dir = DRIVE_ROOT / "models" / model_name
runtime_model_dir = Path("/content/models") / model_name

# Fast local NVMe cache in Colab runtime if available on Drive
if IN_COLAB and drive_model_dir.is_dir() and not (runtime_model_dir / "config.json").is_file():
    print(f"Copying model from Google Drive to local fast runtime storage: {runtime_model_dir}...")
    shutil.copytree(drive_model_dir, runtime_model_dir)

MODEL_PATH = str(runtime_model_dir if runtime_model_dir.is_dir() else (drive_model_dir if drive_model_dir.is_dir() else profile["repo_id"]))
print("Using model location:", MODEL_PATH)

wrapper = load_model(ctx, model_path=MODEL_PATH, device="cuda", local_files_only=Path(MODEL_PATH).is_dir())
prepared = prepare(wrapper, ctx)

print(f"✓ Model loaded: {wrapper.identity['family']} ({wrapper.identity['repo_id']} @ {wrapper.identity['revision']})")
print(f"✓ Prepared {len(prepared)} evaluation prompts with two-shot template.")

## 8. Optional Real-Model IG + KL Checkpoint Equivalence Check

Set `RUN_REAL_MODEL_PREFLIGHT=True` to compare independent B=256 searches against the B=256 snapshots of continuous B=512 searches for both methods, including node values, selected states and AB posteriors. This uses the same reward settings as the full run and costs additional forwards. It is disabled by default; CPU regression tests run in Section 4.


In [ ]:
RUN_REAL_MODEL_PREFLIGHT = False

if RUN_REAL_MODEL_PREFLIGHT:
    from dlm_project.tasks.gsm8k.experiment_1 import execute_search_checkpointed
    low, high = min(BUDGETS), max(BUDGETS)
    for method in METHODS:
        standalone, _ = execute_search_checkpointed(
            wrapper, prepared[0], ctx, method, budgets=(low,),
            max_depth=MAX_DEPTH, num_candidates=MIN_C,
        )
        continuous, _ = execute_search_checkpointed(
            wrapper, prepared[0], ctx, method, budgets=(low, high),
            max_depth=MAX_DEPTH, num_candidates=MIN_C,
        )
        left, right = standalone.checkpoints[low], continuous.checkpoints[low]
        assert left["selected_state_hash"] == right["selected_state_hash"]
        assert left["shortlist_node_ids"] == right["shortlist_node_ids"]
        assert left["node_rewards"] == right["node_rewards"]
        assert len(left["nodes"]) == len(right["nodes"])
        for a, b in zip(left["nodes"], right["nodes"]):
            assert (a.node_id, a.visits, a.reward_sum, a.mean_reward) == (
                b.node_id, b.visits, b.reward_sum, b.mean_reward)
        if method == "ab_medal":
            assert left["posteriors"] == right["posteriors"]
        assert left["reward_metadata"]["kl_extra_model_calls"] == 0
        print(f"{method}: IG + KL checkpoint {low} matches the independent run.")
else:
    print("Optional real-model preflight skipped; enable above if needed.")


## 9. Execute Full Experiment 1 (Evaluation (20 Questions))
Execute both MEDAL and AB-MEDAL across questions at budgets 256, 512 with checkpointing, minimum-C=3 selection, completion caching, real-time per-budget progress logging, incremental per-question saving to Drive (with `os.sync()`), and automatic session resumption (`resume=True`).

In [ ]:
from time import perf_counter
from dlm_project.tasks.gsm8k.experiment_1 import run_e1_stage

print(f"Starting E1 execution on {len(ctx['examples'])} questions for {METHODS} at budgets {BUDGETS}...")
t_start = perf_counter()

# Note: resume=True automatically restores completed questions from disk if your Colab session disconnected.
# Results are incrementally saved and synced to Drive after every question.
q_rows, agg_rows = run_e1_stage(
    wrapper, ctx, OUTPUT_DIR,
    methods=tuple(METHODS),
    budgets=tuple(BUDGETS),
    max_depth=MAX_DEPTH,
    num_candidates=MIN_C,
    resume=True,
)

t_total = perf_counter() - t_start
print(f"\n✓ Experiment 1 Completed in {t_total:.1f}s ({t_total/60:.2f} minutes).")
print(f"✓ All tree checkpoints, completions, and logs saved to: {OUTPUT_DIR / STAGE}")


## 10. Experiment 1 Results & Paired Comparison
Display aggregate accuracy and efficiency metrics across budgets, followed by paired per-question outcomes.

In [ ]:
import pandas as pd
from IPython.display import display

agg_file = OUTPUT_DIR / "summaries" / f"e1_{STAGE}_aggregate.csv"
if not agg_file.is_file():
    agg_file = OUTPUT_DIR / "summaries" / "e1_aggregate.csv"

df_agg = pd.read_csv(agg_file)
print("=== Table 1: E1 Aggregate Performance & Efficiency Summary ===")
display(df_agg[[
    "search_budget", "method", "sample_count", "accuracy_percent",
    "ci_95_low", "ci_95_high", "mean_total_nfe", "mean_selected_depth",
    "mean_search_wall_seconds", "mean_completion_wall_seconds", "mean_total_wall_seconds", "fallback_rate"
]])

pq_file = OUTPUT_DIR / "summaries" / f"e1_{STAGE}_per_question.csv"
if not pq_file.is_file():
    pq_file = OUTPUT_DIR / "summaries" / "e1_per_question.csv"

df_pq = pd.read_csv(pq_file)
print("\n=== Table 2: Paired Per-Question Outcomes (Sample) ===")
display(df_pq[[
    "example_id", "search_budget", "method", "exact_match",
    "predicted_answer", "reference_answer", "selected_depth",
    "search_nfes", "completion_nfes", "total_nfes"
]].head(20))

print("\n=== Reward B diagnostics for selected states ===")
display(df_pq[[
    "example_id", "method", "search_budget", "reward_mode", "kl_weight", "kl_top_k",
    "selected_ig_reward", "selected_action_history_kl", "selected_kl_penalty",
    "peak_history_cache_bytes", "kl_extra_model_calls",
]])


## 11. Run Offline E3 Analysis & Display Publication Figures
Run 100% offline tree and compute-allocation analysis directly on saved E1 traces, generating summary metrics and all 6 publication figures.

Negative IG statistics still measure **raw entropy gain**, not the combined penalized reward. The combined reward and its KL components are stored separately in evaluation traces, checkpoint trees and the reward diagnostics table.

In [ ]:
from dlm_project.tasks.gsm8k.e3_analysis import analyze_e3_stage
from IPython.display import Image, display, Markdown

print("Running E3 Offline Tree Allocation & Topology Analysis...")
e3_q, e3_agg = analyze_e3_stage(
    results_dir=OUTPUT_DIR / STAGE,
    output_dir=OUTPUT_DIR / "summaries",
    stage=STAGE,
    methods=tuple(METHODS),
    budgets=tuple(BUDGETS),
)
print("✓ E3 Analysis Completed successfully.")

figures_dir = OUTPUT_DIR / "summaries" / "e3_figures"
fig_titles = {
    "fig1_nfe_by_depth": "Figure 1: NFE Allocation by Tree Depth — MEDAL vs AB-MEDAL across budgets",
    "fig2_branching_factors": "Figure 2: Branching Factor Distribution — Children per expanded parent",
    "fig3_candidate_density": "Figure 3: Candidate Density vs Depth — Minimum-C availability and selection depth",
    "fig4_root_subtrees": "Figure 4: Root-Subtree Compute Concentration — Search effort across root branches",
    "fig5_reward_redundancy": "Figure 5: Reward and Search Redundancy — Negative IG rates and duplicate states",
    "fig6_accuracy_vs_budget": "Figure 6: Accuracy vs Search Budget — Performance scaling curve",
}

for fig_key, title in fig_titles.items():
    png_path = figures_dir / f"{fig_key}.png"
    if png_path.is_file():
        display(Markdown(f"### {title}"))
        display(Image(filename=str(png_path)))


## 12. Interactive Search Tree & Decision Trace Inspector
Examine candidate shortlists, selection gap $\Delta_d = d_{\max} - d^*$, and Student-t GEN/CONT posterior traces for any question.

In [ ]:
import json
from pathlib import Path

INSPECT_QUESTION_IDX = 0
INSPECT_BUDGET = 256

# Ensure Google Drive is mounted if running in Colab
if Path("/content").is_dir() and not Path("/content/drive/MyDrive").is_dir():
    print("[Notice] Mounting Google Drive to access saved results...")
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print(f"  Could not auto-mount Google Drive: {e}")

# Do not fall back to the original IG experiment's directory.
if "OUTPUT_DIR" not in globals():
    raise RuntimeError("Run Sections 5 and 6 to select this IG + KL run directory.")
actual_output_dir = OUTPUT_DIR

q_info = ctx["examples"][INSPECT_QUESTION_IDX]
ex_id = q_info["example_id"]
print(f"=== Inspecting Question {ex_id} (Budget {INSPECT_BUDGET}) ===")
print("Question:\n", q_info["question"])
print("Reference Answer:", q_info.get("reference_solution", "").split("####")[-1].strip())
print(f"Results Directory: {actual_output_dir / STAGE} (exists: {(actual_output_dir / STAGE).exists()})")

for m in METHODS:
    chk_dir = actual_output_dir / STAGE / m / ex_id / "checkpoints" / str(INSPECT_BUDGET)
    sel_path = chk_dir / "selection.json"
    met_path = chk_dir / "metrics.json"
    tree_path = chk_dir / "tree.json"

    if sel_path.is_file() or met_path.is_file():
        sel_record = json.loads(sel_path.read_text()) if sel_path.is_file() else {}
        met_record = json.loads(met_path.read_text()) if met_path.is_file() else {}

        selected_id = sel_record.get("selected_node_id", met_record.get("selected_node_id"))
        selected_depth = sel_record.get("selected_depth", met_record.get("selected_depth"))

        selected_q = met_record.get("selected_q", sel_record.get("selected_q"))
        if selected_q is None and "node_statistics" in sel_record:
            selected_q = next((s["q_value"] for s in sel_record["node_statistics"] if s["node_id"] == selected_id), None)

        max_depth = met_record.get("max_depth_reached", sel_record.get("max_depth_reached", "N/A"))
        gap = met_record.get("selection_depth_gap", sel_record.get("selection_depth_gap", "N/A"))

        distinct_by_depth = met_record.get("distinct_candidates_by_depth") or sel_record.get("distinct_candidates_by_depth")
        if not distinct_by_depth and tree_path.is_file():
            tree_data = json.loads(tree_path.read_text())
            counts = {}
            for n in tree_data.get("nodes", []):
                counts.setdefault(n["depth"], set()).add(n.get("state_hash", n["node_id"]))
            distinct_by_depth = {d: len(s) for d, s in sorted(counts.items())}

        shortlist_ids = sel_record.get("shortlist_ids") or met_record.get("shortlist_node_ids", [])

        print(f"\n--- {m.upper()} Selection Record ---")
        print(f"  Selected Node: {selected_id} | Depth: {selected_depth}")
        print(f"  Selected Q: {selected_q:.4f}" if selected_q is not None else "  Selected Q: N/A")
        print(f"  Max Depth Reached: {max_depth} | Selection Gap (Δd): {gap}")
        print(f"  Distinct Candidates by Depth: {distinct_by_depth}")
        print(f"  Shortlist Node IDs: {shortlist_ids}")
        print("  Reward:", met_record.get("reward_mode"), "lambda =", met_record.get("kl_weight"))
        print("  Selected IG / history KL / penalty:",
              met_record.get("selected_ig_reward"),
              met_record.get("selected_action_history_kl"),
              met_record.get("selected_kl_penalty"))
    else:
        print(f"\n--- {m.upper()} Selection Record ---")
        print(f"  [Not Found] Expected path: {sel_path}")
        parent = chk_dir.parent
        if not parent.exists():
            print(f"  Parent directory does not exist: {parent}")
        elif parent.is_dir():
            print(f"  Available checkpoints for {ex_id}: {[p.name for p in parent.iterdir()]}")

# For AB-MEDAL, inspect decision events
dec_path = actual_output_dir / STAGE / "ab_medal" / ex_id / "decisions.jsonl"
if dec_path.is_file():
    dec_events = [json.loads(line) for line in dec_path.read_text().splitlines()]
    print(f"\n--- AB-MEDAL Decision Trace Sample (first 10 of {len(dec_events)} decisions) ---")
    for d in dec_events[:10]:
        print(f"  Cycle {d.get('cycle')}: parent node {d.get('node_id')} -> {d.get('choice', d.get('decision_type'))} (forced={d.get('forced')})")
else:
    print(f"\n--- AB-MEDAL Decision Trace Sample ---")
    print(f"  [Not Found] Decision trace not found at: {dec_path}")


## 13. Synchronize All Artifacts to Google Drive
Explicitly flush and sync all in-memory filesystem buffers to Google Drive cloud storage so all traces, trees, completions, CSVs, and plots are safely persisted and reconstructible even if Colab disconnects.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.flush_and_unmount()
    drive.mount("/content/drive")
    print("✓ Google Drive synchronization complete. All artifacts safely flushed to Drive.")
else:
    print(f"✓ Local artifacts safely saved at: {OUTPUT_DIR}")